# **Problem Statement**

## Business Context

A sales forecast is a prediction of future sales revenue based on historical data, industry trends, and the status of the current sales pipeline. Businesses use the sales forecast to estimate weekly, monthly, quarterly, and annual sales totals. A company needs to make an accurate sales forecast as it adds value across an organization and helps the different verticals to chalk out their future course of action.

Forecasting helps an organization plan its sales operations by region and provides valuable insights to the supply chain team regarding the procurement of goods and materials. An accurate sales forecast process has many benefits which include improved decision-making about the future and reduction of sales pipeline and forecast risks. Moreover, it helps to reduce the time spent in planning territory coverage and establish benchmarks that can be used to assess trends in the future.

## Objective

SuperKart is a retail chain operating supermarkets and food marts across various tier cities, offering a wide range of products. To optimize its inventory management and make informed decisions around regional sales strategies, SuperKart wants to accurately forecast the sales revenue of its outlets for the upcoming quarter.

To operationalize these insights at scale, the company has partnered with a data science firm—not just to build a predictive model based on historical sales data, but to develop and deploy a robust forecasting solution that can be integrated into SuperKart’s decision-making systems and used across its network of stores.

## Data Description

The data contains the different attributes of the various products and stores.The detailed data dictionary is given below.

- **Product_Id** - unique identifier of each product, each identifier having two letters at the beginning followed by a number.
- **Product_Weight** - weight of each product
- **Product_Sugar_Content** - sugar content of each product like low sugar, regular and no sugar
- **Product_Allocated_Area** - ratio of the allocated display area of each product to the total display area of all the products in a store
- **Product_Type** - broad category for each product like meat, snack foods, hard drinks, dairy, canned, soft drinks, health and hygiene, baking goods, bread, breakfast, frozen foods, fruits and vegetables, household, seafood, starchy foods, others
- **Product_MRP** - maximum retail price of each product
- **Store_Id** - unique identifier of each store
- **Store_Establishment_Year** - year in which the store was established
- **Store_Size** - size of the store depending on sq. feet like high, medium and low
- **Store_Location_City_Type** - type of city in which the store is located like Tier 1, Tier 2 and Tier 3. Tier 1 consists of cities where the standard of living is comparatively higher than its Tier 2 and Tier 3 counterparts.
- **Store_Type** - type of store depending on the products that are being sold there like Departmental Store, Supermarket Type 1, Supermarket Type 2 and Food Mart
- **Product_Store_Sales_Total** - total revenue generated by the sale of that particular product in that particular store


# **Installing and Importing the necessary libraries**

In [ ]:
# Dependencies are declared in pyproject.toml. For the full project environment run:
# python -m pip install -e ".[api,frontend,notebook,dev]"
# Preprocessing and prediction are packaged together to keep training and inference consistent.

**Note:**

- After running the above cell, kindly restart the notebook kernel (for Jupyter Notebook) or runtime (for Google Colab) and run all cells sequentially from the next cell.

- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in this notebook.

In [ ]:
import warnings
from pathlib import Path
import sys
import os
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    mean_absolute_percentage_error,
)
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import GridSearchCV
import joblib

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Find the project root regardless of whether Jupyter starts here or at the workspace root.
_candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(path for path in _candidates if (path / "data" / "SuperKart.csv").exists())
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from superkart.features import (
    MODEL_FEATURE_COLUMNS, NUMERIC_FEATURES, CATEGORICAL_FEATURES,
    RAW_FEATURE_COLUMNS, TARGET_COLUMN, engineer_features,
)
from superkart.modeling import make_preprocessor, regression_metrics

RANDOM_STATE = 42

# **Loading the dataset**

In [ ]:
data_path = PROJECT_ROOT / "data" / "SuperKart.csv"
df = pd.read_csv(data_path)
print(f"Dataset shape: {df.shape}")
display(df.head())

# **Data Overview**

In [ ]:
print("Rows, columns:", df.shape)
print("Duplicate rows:", df.duplicated().sum())
display(pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_pct": (100 * df.isna().mean()).round(2),
    "unique": df.nunique(),
}))
display(df.describe(include="all").T)

# **Exploratory Data Analysis (EDA)**

## Univariate Analysis

In [ ]:
numeric_columns = df.select_dtypes(include=np.number).columns.tolist()
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df[TARGET_COLUMN], bins=40, kde=True, ax=axes[0], color="#187a72")
axes[0].set_title("Product-store sales distribution")
sns.boxplot(x=df[TARGET_COLUMN], ax=axes[1], color="#e8a03c")
axes[1].set_title("Sales outlier overview")
plt.tight_layout()
plt.show()

for column in ["Product_Sugar_Content", "Store_Size", "Store_Location_City_Type", "Store_Type"]:
    display(df[column].value_counts(dropna=False).rename_axis(column).to_frame("count"))

## Bivariate Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.boxplot(data=df, x="Store_Type", y=TARGET_COLUMN, ax=axes[0], color="#4c9f70")
axes[0].tick_params(axis="x", rotation=25)
axes[0].set_title("Sales by store type")
sns.scatterplot(data=df.sample(min(1500, len(df)), random_state=RANDOM_STATE),
                x="Product_MRP", y=TARGET_COLUMN, hue="Store_Location_City_Type",
                alpha=0.55, ax=axes[1])
axes[1].set_title("Sales vs product MRP by city tier")
plt.tight_layout()
plt.show()

store_summary = df.groupby("Store_Type", observed=True)[TARGET_COLUMN].agg(["count", "mean", "median"])
city_summary = df.groupby("Store_Location_City_Type", observed=True)[TARGET_COLUMN].agg(["count", "mean", "median"])
display(store_summary.sort_values("mean", ascending=False))
display(city_summary.sort_values("mean", ascending=False))
print("Highest mean sales store type:", store_summary["mean"].idxmax())
print("Product MRP / sales correlation:", round(df[["Product_MRP", TARGET_COLUMN]].corr().iloc[0, 1], 3))
category_summary = engineer_features(df).assign(Sales=df[TARGET_COLUMN].to_numpy()).groupby(
    "Product_Type_Category", observed=True
)["Sales"].agg(["count", "mean", "median"])
display(category_summary.sort_values("mean", ascending=False))

# **Data Preprocessing**

In [ ]:
# Derive the same ten predictors used by the online and batch deployment interfaces.
model_df = df.dropna(subset=[TARGET_COLUMN]).copy()
X = engineer_features(model_df)
y = pd.to_numeric(model_df[TARGET_COLUMN], errors="coerce")
valid_rows = y.notna()
X, y = X.loc[valid_rows], y.loc[valid_rows]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
preprocessor = make_preprocessor()
print("Training shape:", X_train.shape, "Test shape:", X_test.shape)
print("Model features:", MODEL_FEATURE_COLUMNS)
print("Missing engineered values:", int(X.isna().sum().sum()))

# **Model Building**

## Define functions for Model Evaluation

In [ ]:
# function to compute adjusted R-squared
def adj_r2_score(predictors, targets, predictions):
    r2 = r2_score(targets, predictions)
    n = predictors.shape[0]
    k = predictors.shape[1]
    return 1 - ((1 - r2) * (n - 1) / (n - k - 1))


# function to compute different metrics to check performance of a regression model
def model_performance_regression(model, predictors, target):
    """
    Function to compute different metrics to check regression model performance

    model: regressor
    predictors: independent variables
    target: dependent variable
    """

    # predicting using the independent variables
    pred = model.predict(predictors)

    r2 = r2_score(target, pred)  # to compute R-squared
    adjr2 = adj_r2_score(predictors, target, pred)  # to compute adjusted R-squared
    rmse = np.sqrt(mean_squared_error(target, pred))  # to compute RMSE
    mae = mean_absolute_error(target, pred)  # to compute MAE
    mape = mean_absolute_percentage_error(target, pred)  # to compute MAPE

    # creating a dataframe of metrics
    df_perf = pd.DataFrame(
        {
            "RMSE": rmse,
            "MAE": mae,
            "R-squared": r2,
            "Adj. R-squared": adjr2,
            "MAPE": mape,
        },
        index=[0],
    )

    return df_perf

The ML models to be built can be any two out of the following:
1. Decision Tree
2. Bagging
3. Random Forest
4. AdaBoost
5. Gradient Boosting
6. XGBoost

In [ ]:
models = {
    "Random Forest": Pipeline([
        ("preprocessor", make_preprocessor()),
        ("regressor", RandomForestRegressor(
            n_estimators=160, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1
        )),
    ]),
    "Gradient Boosting": Pipeline([
        ("preprocessor", make_preprocessor()),
        ("regressor", GradientBoostingRegressor(
            n_estimators=120, max_depth=2, learning_rate=0.06, random_state=RANDOM_STATE
        )),
    ]),
}
model_results = {}
for model_name, model in models.items():
    model.fit(X_train, y_train)
    model_results[model_name] = regression_metrics(model, X_test, y_test)
performance = pd.DataFrame(model_results).T.sort_values("RMSE")
display(performance)
print("Baseline model comparison uses the same held-out test split.")

# **Model Performance Improvement - Hyperparameter Tuning**

In [ ]:
# Tune Random Forest with a compact cross-validated grid; preprocessing is fitted inside each fold.
parameter_grid = {
    "regressor__n_estimators": [120, 200],
    "regressor__max_depth": [None, 16],
    "regressor__min_samples_leaf": [1, 2],
}
tuning_search = GridSearchCV(
    Pipeline([
        ("preprocessor", make_preprocessor()),
        ("regressor", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)),
    ]),
    parameter_grid,
    scoring="neg_root_mean_squared_error",
    cv=3,
    n_jobs=1,
    refit=True,
)
tuning_search.fit(X_train, y_train)
tuned_model = tuning_search.best_estimator_
tuned_metrics = regression_metrics(tuned_model, X_test, y_test)
print("Best parameters:", tuning_search.best_params_)
print("Tuned held-out metrics:")
display(pd.DataFrame([tuned_metrics], index=["Tuned Random Forest"]))

# **Model Performance Comparison, Final Model Selection, and Serialization**

In [ ]:
comparison = pd.DataFrame(model_results).T
comparison.loc["Tuned Random Forest"] = tuned_metrics
comparison = comparison.sort_values("RMSE")
display(comparison)
selected_name = comparison["RMSE"].idxmin()
selected_model = tuned_model if selected_name == "Tuned Random Forest" else models[selected_name]
artifact_path = PROJECT_ROOT / "backend_files" / "superkart_model.joblib"
artifact_path.parent.mkdir(parents=True, exist_ok=True)
model_artifact = {
    "model": selected_model,
    "model_name": selected_name,
    "feature_columns": MODEL_FEATURE_COLUMNS,
    "metrics": comparison.to_dict(orient="index"),
    "best_params": tuning_search.best_params_,
}
joblib.dump(model_artifact, artifact_path)
print(f"Selected {selected_name}; serialized model at {artifact_path}")

# **Deployment - Backend**

## Points to note before executing the below cells
- Create a GitHub account if you don't already have one at [github.com](https://github.com)
- Generate a **Personal Access Token**
  - Make sure the token has **repo** scope (full control of private repositories)
- The serialized ML model file (`superkart_model.joblib`) should already be present in the `backend_files` folder before pushing to GitHub
- We will deploy **both the Flask backend and the Streamlit frontend as separate Docker containers** inside a GitHub Codespace, and connect them using a **Docker network**

## Flask Web Framework

Flask is a lightweight, flexible Python web framework used to build web applications and APIs quickly and easily.

Flask allows you to:
- Create web routes (URLs that users can access)
- Handle HTTP requests and responses
- Build REST APIs to expose machine learning models or other services

In [ ]:
# Flask API source: POST /v1/predict accepts JSON; POST /v1/predictbatch accepts a CSV upload.
print((PROJECT_ROOT / "src" / "superkart" / "api.py").read_text(encoding="utf-8"))

## Dependencies File

In [ ]:
import tomllib
project_metadata = tomllib.loads((PROJECT_ROOT / "pyproject.toml").read_text(encoding="utf-8"))
print("Install with: python -m pip install -e .")
print("Runtime dependencies:")
for dependency in project_metadata["project"]["dependencies"]:
    print(" -", dependency)

## Dockerfile

In [ ]:
display((PROJECT_ROOT / "Dockerfile.backend").read_text(encoding="utf-8"))

# **Deployment - Frontend**

## Streamlit for Interactive UI

In [ ]:
display((PROJECT_ROOT / "frontend" / "app.py").read_text(encoding="utf-8"))

## Dependencies File

In [ ]:
# Runtime dependencies and versions are declared centrally in pyproject.toml.
display((PROJECT_ROOT / "pyproject.toml").read_text(encoding="utf-8"))

## Dockerfile

In [ ]:
# The backend image copies the project package and serialized artifact.
display((PROJECT_ROOT / "Dockerfile.backend").read_text(encoding="utf-8"))

# **Pushing Deployment Files to GitHub**

In [ ]:
# Confirm deployment files are ready; publishing requires a user-owned GitHub remote.
for relative_path in ["docker-compose.yml", "Dockerfile.backend", "Dockerfile.frontend", ".devcontainer/devcontainer.json"]:
    print(relative_path, "exists:", (PROJECT_ROOT / relative_path).exists())

# **Inferencing using Flask API**

This API is implemented in `src/superkart/api.py`. The model artifact is created by the training section and loaded from `backend_files/superkart_model.joblib`. The notebook's API examples below are optional and only run when `SUPERKART_API_URL` is set.

For deployment, build and run the included Docker Compose stack from the project root. In Codespaces, forward port 7860 for the API and 8501 for Streamlit; make the API port Public only when external clients need it. Configure GitHub remotes and authentication outside the notebook, and never commit personal access tokens.

In [ ]:
import requests


In [ ]:
model_root_url = os.getenv("SUPERKART_API_URL", "").rstrip("/")
print("Configured API:", model_root_url or "not configured; live inference cells will be skipped")

Replace the URL above with the actual forwarded URL from your Codespace's Ports tab. It will look something like `https://organic-space-abcd1234-7860.app.github.dev`. Make sure there is no trailing slash at the end.

This URL is a tunnel that Codespaces creates to route external HTTP requests into your Codespace and to the Flask container running on port 7860.

In [ ]:
model_url = model_root_url + "/v1/predict"  # Endpoint for online (single) inference

Since our model predictions are provided by the following endpoint we created using Flask, we need to invoke the same to make prediction.

> ```@superkart_api.post('/v1/predict')```

In [ ]:
model_batch_url = model_root_url + "/v1/predictbatch"  # Endpoint for batch inference

> ```@superkart_api.post('/v1/predictbatch')```

## Online

**Online Inference:** Sending a single request to the API and receiving an immediate response. This is useful for real-time applications like recommendation systems and fraud detection.  
* This data is sent as a JSON payload in a POST request to the model endpoint.

* The model processes the input features and returns a prediction.

In [ ]:
payload = {
  "Product_Weight": 12.66,
  "Product_Sugar_Content": "Low Sugar",
  "Product_Allocated_Area": 0.027,
  "Product_MRP": 117.08,
  "Store_Size": "Medium",
  "Store_Location_City_Type": "Tier 2",
  "Store_Type": "Supermarket Type2",
  "Product_Id_char": "FD",
  "Store_Age_Years": 16,
  "Product_Type_Category": "Non Perishables"
}

In [ ]:
if model_root_url:
    response = requests.post(model_url, json=payload, timeout=20)
    response.raise_for_status()
else:
    response = None
    print("Skipped online request: set SUPERKART_API_URL to a running API URL.")

In [ ]:
response if response is not None else "No live API response in this local run."

`<Response [200]>` indicates that the HTTP request was successful.  

- `Response` is the object returned by `requests.post()`.  
- `[200]` is the HTTP status code, meaning **OK** (the request was processed successfully).  

In [ ]:
if response is not None:
    print(response.json())

`response.json()` is used to parse the response body as JSON.  
- If the API returns data in JSON format, `.json()` converts it into a Python dictionary.  
- This allows easy access to specific values using keys.  

## Batch

**Batch Inference:** Sending multiple inputs in a single request, allowing efficient processing of large datasets. This is ideal for analyzing historical data at scale.

In [ ]:
batch_dataset = pd.read_csv(PROJECT_ROOT / "data" / "Batch_Data_SuperKart.csv")
display(batch_dataset.head())

**Note**: The batch CSV file must contain all the feature columns expected by the model: `Product_Weight`, `Product_Sugar_Content`, `Product_Allocated_Area`, `Product_MRP`, `Store_Size`, `Store_Location_City_Type`, `Store_Type`, `Product_Id_char`, `Store_Age_Years`, `Product_Type_Category`.

In [ ]:
batch_input = {"file": ("batch.csv", batch_dataset.to_csv(index=False).encode("utf-8"), "text/csv")}
print("Batch rows:", len(batch_dataset))

This code prepares the product data as a **file-like object** to send in an API request (in the `files` parameter of `requests.post`).

- `batch_dataset`: This is a Pandas DataFrame containing the product data loaded from a CSV file.

- `.to_csv(header=True, index=False)`: Converts the DataFrame into a CSV **string**, keeping the column headers (`header=True`) but **excluding the index** (`index=False`).

- `.encode('utf-8')`: Encodes the CSV string into **bytes** (UTF-8 format), which is necessary for sending it over an HTTP request.

- `'file': ...`: Wraps the encoded CSV data in a dictionary with the key `'file'`, which is expected by the Flask backend.

In [ ]:
if model_root_url:
    response = requests.post(model_batch_url, files={"file": batch_input["file"]}, timeout=60)
    response.raise_for_status()
else:
    response = None
    print("Skipped batch request: set SUPERKART_API_URL to a running API URL.")

This line sends a **POST request** to the deployed model API to perform **batch sales prediction**.

- `requests.post(...)`: This uses the `requests` library to make a **POST** request to a web server (in this case, our API endpoint for batch predictions).

- `model_batch_url`: This is the URL endpoint where the batch prediction API is hosted. It should look like this:\
  `"https://<your-codespace-name>-7860.app.github.dev/v1/predictbatch"`

- `files=batch_input`: This sends the batch data file (e.g., a CSV) as part of the request using the `files` parameter.

- `response`: This is the result of the API call and contains the response from the server, including predicted sales values.

In [ ]:
response if response is not None else "No live API response in this local run."

In [ ]:
if response is not None:
    display(response.json())

As we can see, we receive a JSON where each key represents a row index, and the value represents the model's predicted sales for that product.

# **Actionable Insights and Business Recommendations**

- Use the held-out RMSE and MAE to forecast revenue by product and store, and pilot replenishment recommendations in a limited set of outlets before scaling. Reassess the model on a time-based holdout when future-period data is available; the current random split is a baseline, not proof of future-quarter accuracy.

- Compare predicted sales by store type, city tier, and product category alongside the EDA summaries. Prioritize stock allocation using both expected revenue and operational constraints, monitor prediction error by segment, and retrain when store conditions or product mix materially change. Treat forecasts as decision support rather than guaranteed sales.